# Notebook 02 Data Wrangling & Preprocessing: Curhat NLP
**Proyek Analisis Data: HAPI (Human Activity Pattern Intelligence)**

### Profil Proyek & Tim
- **Tema Capstone:** Healthy Lives & Well-being
- **Target User:** Mahasiswa (fokus pada aspek akademik & lingkungan belajar)
- **Tujuan Proyek:** Pengembangan *platform* Webapp untuk *mood management & tracker* serta diagnosis mandiri tingkat kelelahan (*fatigue*).
- **Tim DS:** Greycia Febrina Michelle (CDCC700D6X2644) & Khazel Hayfa Yosmi (CDCC308D6X0629)

### Tujuan Notebook
Melakukan proses Data Wrangling secara *end-to-end* (Gathering → Assessing → Cleaning) dan Preprocessing teks pada dataset Curhat NLP. Langkah ini bertujuan untuk menghasilkan data teks yang bersih, terstandardisasi, dan siap digunakan oleh AI Engineer untuk melatih model klasifikasi emosi.

### Peran Dataset
Dataset yang digunakan dalam tahapan ini bersumber dari Kaggle: [Indonesian Student Vents](https://www.kaggle.com/datasets/cereycie/indonesian-student-vents).

Dataset ini berisi teks curhatan mahasiswa Indonesia dalam bahasa sehari-hari (informal) beserta label emosi dominannya. 

Data yang telah diproses nantinya akan digunakan untuk melatih model NLP yang memproses input dari **fitur Curhat AI** pada aplikasi HAPI. Model ini berfungsi untuk mengenali dan memetakan kondisi emosional pengguna melalui teks yang mereka masukkan.

### Spesifikasi Kolom Dataset
- `student_id`: kode pengenal (identifier) unik mahasiswa, tidak dimasukkan ke dalam model.
- `text_curhat`: fitur (X) berupa input teks cerita atau keluhan mahasiswa yang akan diproses menggunakan teknik NLP.
- `emotion`: target (y) berupa label emosi dominan dengan kategori: Sadness / Anger / Fear / Joy / Neutral.
- `row_status`: kolom keperluan audit data yang akan dihapus setelah proses cleaning selesai.

### Catatan & Limitasi Dataset
- Dataset ini tidak mengandung `fatigue_score` karena fatigue bukan merupakan output dari model NLP. Model NLP bertugas khusus untuk mengklasifikasi emosi, bukan memprediksi fatigue.
- Estimasi fatigue dari teks merupakan tugas dari **Fusion Model** yang akan dikerjakan oleh AI Engineer pada tahapan terpisah.
- Penggunaan gaya bahasa sarkasme oleh pengguna merupakan batasan yang diketahui (*known limitation*) dan telah didokumentasikan dalam pengembangan proyek ini.

### Referensi Penelitian
- Guntuku, S. C., Yaden, D. B., Kern, M. L., Ungar, L. H., & Eichstaedt, J. C. (2017). Detecting depression and mental illness on social media: an integrative review. Current Opinion in Behavioral Sciences, 18, 43-49.

## Setup

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / 'data').exists():
        break
    ROOT = ROOT.parent

RAW_PATH   = ROOT / 'data' / 'raw' / 'model_ready' / 'curhat_nlp_dirty.csv'
CLEAN_PATH = ROOT / 'data' / 'clean' / 'model_ready' / 'curhat_nlp_clean.csv'
PREP_PATH  = ROOT / 'data' / 'preprocessed' / 'curhat_nlp_preprocessed.csv'

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
PREP_PATH.parent.mkdir(parents=True, exist_ok=True)

VALID_EMOTIONS = {'Sadness', 'Anger', 'Fear', 'Joy', 'Neutral'}

STRESS_KEYWORDS = [
    'capek', 'lelah', 'stres', 'burnout', 'deadline', 'revisi', 'skripsi',
    'takut', 'cemas', 'panik', 'nangis', 'down', 'menyerah', 'buntu',
    'overthinking', 'gelisah', 'kewalahan', 'frustasi', 'tertekan'
]

print(f'ROOT     : {ROOT}')
print(f'File ada : {RAW_PATH.exists()}')
print('Setup selesai.')

ROOT     : d:\Proyek_Analisis_Burnout
File ada : True
Setup selesai.


**Dokumentasi Setup Data: Curhat NLP**  
**Proyek HAPI (Human Activity Pattern Intelligence)**

Kami menyusun script ini untuk menyiapkan fondasi pembersihan data teks curhatan mahasiswa. Tantangan utamanya adalah menangani data teks yang tidak terstruktur, jadi kami perlu menetapkan parameter awal sebelum masuk ke tahap pembersihan yang lebih dalam.

**Operasi Utama**

**Path Mapping**  
Kami menggunakan `pathlib` agar script tetap berjalan stabil saat berpindah folder kerja. Tidak perlu lagi pusing dengan *hardcoded path* yang sering error saat berpindah perangkat.

**Directory Setup**  
Kami membuat folder penyimpanan secara otomatis jika belum tersedia. Kami membagi alur data menjadi tahap `raw`, `clean`, dan `preprocessed` agar proses pipeline tetap rapi.

**Standarisasi & Validasi**  
Kami sengaja membatasi `max_colwidth` sampai **120 karakter** agar saat pengecekan data nanti teks curhatan tidak terpotong. Kami juga mendefinisikan `VALID_EMOTIONS` sebagai filter target serta `STRESS_KEYWORDS` untuk menangkap istilah slang mahasiswa di Indonesia.

**Alur Kerja Data**

```text
Curhat Mentah
      │
      ▼
   RAW_PATH
      │
      ▼
 Text Cleaning
      │
      ▼
  CLEAN_PATH
      │
      ▼
Preprocessing NLP
      │
      ▼
 PREP_PATH
      │
      ▼
 Machine Learning
```

**Catatan Teknis**

Setup ini sudah aman, dan langkah berikutnya adalah mulai masuk ke proses cleaning untuk membuang noise pada teks. Kami mungkin akan menambahkan keyword lagi nanti jika dalam eksplorasi data ditemukan istilah slang baru yang sering muncul di curhatan.

## Gathering Data

In [2]:
df_raw = pd.read_csv(RAW_PATH)
print(f'Dataset dimuat: {df_raw.shape[0]:,} baris, {df_raw.shape[1]} kolom')
print(f'Kolom: {df_raw.columns.tolist()}')
df_raw.head()

Dataset dimuat: 12,200 baris, 4 kolom
Kolom: ['student_id', 'text_curhat', 'emotion', 'row_status']


,student_id,text_curhat,emotion,row_status
0,MHS_07126,"Hari ini fokus ke, pergi ke lab komputer jurusan buat mencari referensi jurnal pendukung, biar ga numpuk di akhir mi...",Neutral,clean
1,MHS_00809,"Bener-bener bikin emosi, kerja keras buat revisi yang standarnya terus berubah tapi hasilnya selalu berubah keputusa...",Anger,clean
2,MHS_01690,"Mental udah di ujung, stuck di progress bimbingan minggu ini gara-gara belum nemu solusi di bagian olah data statist...",Sadness,clean
3,MHS_D00001,NaN,Joy,dirty_missing_text
4,MHS_01785,"Rasanya mau meledak, urusan jadwal bimbingan yang tidak konsisten ga kelar-kelar karena hanya di-read tanpa balasan,...",Anger,clean


**Pemuatan Dataset Curhat Mahasiswa**

**Penjelasan Teknis**

- `pd.read_csv(RAW_PATH)`: Membaca file dataset ke dalam dataframe kami.

- `df_raw.shape`: Memastikan berapa banyak data yang kami tangani saat ini.

- `df_raw.columns.tolist()`: Memeriksa daftar kolom untuk memastikan struktur data sudah sesuai dengan rencana.

- `df_raw.head()`: Menampilkan pratinjau data agar kami bisa melihat langsung kualitas teks curhat yang ada.

**Ringkasan Kondisi Data**

Dataset yang kami muat berisi **12.200 baris** dengan **4 kolom utama**. Fokus kami ada pada kolom `text_curhat` dan `emotion` yang menjadi inti dari penelitian ini. Kami juga mencatat adanya kolom `row_status` yang membantu kami melacak data bermasalah, seperti baris dengan nilai kosong (`NaN`) yang muncul di data contoh. Secara keseluruhan, data ini sudah siap kami bedah lebih lanjut untuk melihat pola emosi di balik curhatan tersebut.


## Assessing Data

In [3]:
print('Shape')
print(f'Baris: {df_raw.shape[0]:,} | Kolom: {df_raw.shape[1]}')

Shape
Baris: 12,200 | Kolom: 4


In [4]:
print('Tipe Data')
print(df_raw.dtypes)

Tipe Data
student_id     object
text_curhat    object
emotion        object
row_status     object
dtype: object


In [5]:
print('Missing Values')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
print(pd.DataFrame({'jumlah': missing, 'persen': missing_pct})[missing > 0])

Missing Values
             jumlah  persen
text_curhat     300    2.46
emotion         200    1.64


In [6]:
print('Duplikat')
n_dup = df_raw.duplicated(subset=['text_curhat', 'emotion']).sum()
print(f'Baris duplikat (text + emotion): {n_dup}')

Duplikat
Baris duplikat (text + emotion): 1675


In [7]:
print('Label Emotion')
print(df_raw['emotion'].value_counts(dropna=False))

print(f'\nLabel Tidak Valid (Bukan Salah Satu Dari {VALID_EMOTIONS})')
invalid_emotion = df_raw[~df_raw['emotion'].isin(VALID_EMOTIONS) & df_raw['emotion'].notna()]
print(invalid_emotion['emotion'].value_counts())

Label Emotion
emotion
Sadness    3202
Fear       2914
Anger      2866
Joy        1454
Neutral    1064
Sadnes      270
anger       230
NaN         200
Name: count, dtype: int64

Label Tidak Valid (Bukan Salah Satu Dari {'Anger', 'Neutral', 'Sadness', 'Joy', 'Fear'})
emotion
Sadnes    270
anger     230
Name: count, dtype: int64


In [8]:
print('Whitespace Berlebih')
has_ws = df_raw['text_curhat'].dropna().str.startswith('   ').sum()
print(f'Teks dengan whitespace di awal: {has_ws}')

Whitespace Berlebih
Teks dengan whitespace di awal: 1000


In [9]:
print('Distribusi Row Status')
print(df_raw['row_status'].value_counts())

Distribusi Row Status
row_status
clean                    10000
dirty_whitespace          1000
dirty_missing_text         300
dirty_typo_Sadness         270
dirty_typo_Anger           230
dirty_duplicate            200
dirty_missing_emotion      200
Name: count, dtype: int64


**Profil Dataset**

Kami mengolah **12.200 entri** dengan **4 kolom** yang mencakup ID mahasiswa, teks curhat, label emosi, dan status baris. Secara struktural, tipe data sudah terbaca dengan baik, namun kami menemukan beberapa masalah kualitas yang memerlukan perhatian sebelum tahap pemodelan.

**Temuan Kritis**

Kami mengidentifikasi beberapa titik masalah yang akan mengganggu validitas hasil analisis jika dibiarkan:

- **Data Kosong:** Terdapat celah pada **300 baris** kolom `text_curhat` dan **200 baris** pada kolom `emotion`. Kami perlu memutuskan apakah akan melakukan pembersihan atau imputasi data.

- **Duplikasi:** Terdapat **1.675 baris duplikat** yang identik berdasarkan kombinasi teks dan label emosi. Ini adalah jumlah yang signifikan dan dapat menyebabkan bias pada model.

- **Inkonsistensi Kategori:** Label emosi memiliki variasi penulisan yang tidak konsisten seperti `Sadnes` dan `anger`. Kami perlu menyeragamkan label tersebut agar sesuai dengan kategori valid yang kami tentukan.

- **Format Teks:** Sebanyak **1.000 baris** memiliki spasi berlebih pada awal teks. Kami harus melakukan normalisasi teks agar data seragam dan siap diproses lebih lanjut.

**Ringkasan Hasil Assessing**

| Masalah | Detail | Tindakan |
|----------|----------|----------|
| Missing text_curhat | ~300 baris teks kosong | Hapus baris |
| Missing emotion | ~200 baris tanpa label | Hapus baris |
| Typo label emotion | 'Sadnes', 'anger' | Hapus baris |
| Whitespace berlebih | ~1000 baris dengan spasi di awal/akhir | Strip whitespace |
| Duplikat | ~200 baris duplikat | Hapus baris |
| Kolom tidak perlu | `row_status` | Drop sebelum export |

**Known limitation yang didokumentasikan:**

Dataset ini tidak menangani sarkasme. Kalimat sarkastis (mis. "Seneng banget harus revisi bab 4 lagi") akan salah diklasifikasi sebagai Joy, padahal bermakna negatif. Ini adalah limitasi fundamental dari supervised classification berbasis template sintetis.


## Cleaning Data

In [10]:
df = df_raw.copy()
before = len(df)

df = df[df['row_status'] == 'clean'].copy()
print(f'Drop Baris Kotor: {before - len(df)} dihapus → sisa {len(df):,}')

Drop Baris Kotor: 2200 dihapus → sisa 10,000


In [11]:
df['text_curhat'] = df['text_curhat'].str.strip()
print('Whitespace Di-Strip ✓')

Whitespace Di-Strip ✓


In [12]:
df = df.drop(columns=['row_status'])
print(f'Drop Row Status ✓ | Kolom: {df.columns.tolist()}')

Drop Row Status ✓ | Kolom: ['student_id', 'text_curhat', 'emotion']


In [13]:
assert df['text_curhat'].isnull().sum() == 0
assert df['emotion'].isnull().sum() == 0
assert set(df['emotion'].unique()) == VALID_EMOTIONS

print('Semua Verifikasi Passed ✓')
print(f'Shape Akhir Cleaned: {df.shape}')
df.head(3)

Semua Verifikasi Passed ✓
Shape Akhir Cleaned: (10000, 3)


,student_id,text_curhat,emotion
0,MHS_07126,"Hari ini fokus ke, pergi ke lab komputer jurusan buat mencari referensi jurnal pendukung, biar ga numpuk di akhir mi...",Neutral
1,MHS_00809,"Bener-bener bikin emosi, kerja keras buat revisi yang standarnya terus berubah tapi hasilnya selalu berubah keputusa...",Anger
2,MHS_01690,"Mental udah di ujung, stuck di progress bimbingan minggu ini gara-gara belum nemu solusi di bagian olah data statist...",Sadness


In [14]:
df.to_csv(CLEAN_PATH, index=False)
print(f'Dataset Cleaned Disimpan Ke: {CLEAN_PATH}')

Dataset Cleaned Disimpan Ke: d:\Proyek_Analisis_Burnout\data\clean\model_ready\curhat_nlp_clean.csv


**Pembersihan Data untuk Dataset Curhat**

Kami menyelesaikan pembersihan data untuk teks curhat mahasiswa agar siap digunakan ke tahap NLP. Fokus kami adalah memastikan teks yang masuk ke model sudah bersih dari noise dan memiliki label emosi yang valid.

**Langkah-Langkah Teknis**

- **Penyaringan Data Mentah:** Kami membuang **2.200 baris** yang ditandai sebagai kotor agar dataset hanya berisi data berkualitas untuk pelatihan.

- **Normalisasi Teks:** Kami menjalankan fungsi `strip` pada kolom `text_curhat` untuk memastikan tidak ada spasi tambahan yang mengganggu proses tokenisasi nanti.

- **Pemangkasan Kolom:** Kolom `row_status` kami buang karena fungsinya sudah selesai dan tidak dibutuhkan lagi dalam model NLP.

- **Verifikasi Integritas:** Kami melakukan pengecekan ketat untuk memastikan tidak ada nilai kosong di kolom teks maupun emosi. Kami juga memastikan label emosi yang ada di dataset sesuai dengan daftar valid yang sudah kami tentukan sebelumnya.

**Hasil Akhir**

Setelah proses filter dan verifikasi, kami mendapatkan dataset bersih dengan total **10.000 baris** dan **3 kolom** yaitu ID, teks curhat, dan emosi. File akhir ini sudah kami simpan di `curhat_nlp_clean.csv` dan siap untuk masuk ke tahap ekstraksi fitur atau pemrosesan vektor teks.

## Preprocessing & Feature Engineering

In [15]:
df_prep = pd.read_csv(CLEAN_PATH)

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_prep['text_clean'] = df_prep['text_curhat'].apply(clean_text)

print('Contoh Sebelum Vs Sesudah Preprocessing')
for _, row in df_prep.head(3).iterrows():
    print(f'  ASLI  : {row["text_curhat"][:80]}')
    print(f'  BERSIH: {row["text_clean"][:80]}')
    print()

Contoh Sebelum Vs Sesudah Preprocessing
  ASLI  : Hari ini fokus ke, pergi ke lab komputer jurusan buat mencari referensi jurnal p
  BERSIH: hari ini fokus ke pergi ke lab komputer jurusan buat mencari referensi jurnal pe

  ASLI  : Bener-bener bikin emosi, kerja keras buat revisi yang standarnya terus berubah t
  BERSIH: bener bener bikin emosi kerja keras buat revisi yang standarnya terus berubah ta

  ASLI  : Mental udah di ujung, stuck di progress bimbingan minggu ini gara-gara belum nem
  BERSIH: mental udah di ujung stuck di progress bimbingan minggu ini gara gara belum nemu



### Feature Engineering: Statistik Teks

In [16]:
df_prep['char_count']  = df_prep['text_clean'].str.len()
df_prep['word_count']  = df_prep['text_clean'].str.split().str.len()

STRESS_KEYWORDS = [
    'capek', 'lelah', 'stres', 'burnout', 'deadline', 'revisi', 'skripsi',
    'takut', 'cemas', 'panik', 'nangis', 'down', 'menyerah', 'buntu',
    'overthinking', 'gelisah', 'kewalahan', 'frustasi', 'tertekan'
]

def count_stress_keywords(text: str) -> int:
    return sum(1 for kw in STRESS_KEYWORDS if kw in text)

df_prep['stress_keyword_count'] = df_prep['text_clean'].apply(count_stress_keywords)

print('Statistik Fitur Teks')
print(df_prep[['char_count', 'word_count', 'stress_keyword_count']].describe().round(2))

Statistik Fitur Teks
       char_count  word_count  stress_keyword_count
count    10000.00    10000.00              10000.00
mean       152.16       23.08                  0.79
std         23.08        3.57                  0.80
min         85.00       13.00                  0.00
25%        135.00       21.00                  0.00
50%        153.00       23.00                  1.00
75%        169.00       26.00                  1.00
max        226.00       33.00                  4.00


### Encoding Target dan Leakage Check

In [17]:
emotion_map = {'Neutral': 0, 'Joy': 1, 'Fear': 2, 'Anger': 3, 'Sadness': 4}
df_prep['emotion_encoded'] = df_prep['emotion'].map(emotion_map)

print('Encoding emotion:')
print(df_prep[['emotion', 'emotion_encoded']].value_counts().sort_index())

Encoding emotion:
emotion  emotion_encoded
Anger    3                  2542
Fear     2                  2584
Joy      1                  1157
Neutral  0                   782
Sadness  4                  2935
Name: count, dtype: int64


In [18]:
FEATURE_COLS = ['text_curhat', 'text_clean', 'char_count', 'word_count', 'stress_keyword_count']
TARGET_COLS  = ['emotion', 'emotion_encoded']

leakage = set(FEATURE_COLS) & set(TARGET_COLS)
assert len(leakage) == 0, f'LEAKAGE: {leakage}'
print(f'Leakage check: BERSIH ✓')

Leakage check: BERSIH ✓


In [19]:
FINAL_COLS = ['student_id'] + FEATURE_COLS + TARGET_COLS
df_final = df_prep[FINAL_COLS].copy()

df_final.to_csv(PREP_PATH, index=False)
print(f'Dataset Model-Ready Disimpan Ke: {PREP_PATH}')

print('\nRingkasan Final')
print(f'Total Baris: {len(df_final):,}')
print('Fitur Teks: text_curhat (raw), text_clean (preprocessed)')
print('Fitur Numerik: char_count, word_count, stress_keyword_count')
print('Kolom Target: emotion (label), emotion_encoded (numerik 0–4)')
print(f'Missing Values: {df_final.isnull().sum().sum()}')
print('Leakage: BERSIH')

print('\nDistribusi Emotion')
print(df_final['emotion'].value_counts())

Dataset Model-Ready Disimpan Ke: d:\Proyek_Analisis_Burnout\data\preprocessed\curhat_nlp_preprocessed.csv

Ringkasan Final
Total Baris: 10,000
Fitur Teks: text_curhat (raw), text_clean (preprocessed)
Fitur Numerik: char_count, word_count, stress_keyword_count
Kolom Target: emotion (label), emotion_encoded (numerik 0–4)
Missing Values: 0
Leakage: BERSIH

Distribusi Emotion
emotion
Sadness    2935
Fear       2584
Anger      2542
Joy        1157
Neutral     782
Name: count, dtype: int64


**Rekayasa Fitur Teks untuk Analisis Emosi**

Kami telah merampungkan persiapan data teks untuk kebutuhan pemodelan NLP. Fokus kami adalah mengubah curhatan mentah menjadi format yang kaya fitur namun tetap terjaga integritasnya agar model bisa menangkap pola emosional dengan lebih akurat.

**Verifikasi dan Transformasi Data**

- **Pembersihan Teks:** Kami melakukan normalisasi dengan mengubah teks ke huruf kecil, menghapus karakter non-alfanumerik, serta merapikan spasi. Langkah ini menghasilkan kolom `text_clean` yang seragam untuk diolah lebih lanjut.

- **Ekstraksi Fitur Numerik:** Kami tidak hanya mengandalkan teks mentah. Kami menambahkan fitur pendukung berupa jumlah karakter, jumlah kata, serta skor `stress_keyword_count` yang menghitung kemunculan kata kunci terkait tekanan psikologis.

- **Encoding Target:** Label emosi seperti `Neutral`, `Joy`, `Fear`, `Anger`, dan `Sadness` kami konversi ke format numerik (`0` sampai `4`) agar sesuai dengan kebutuhan algoritma klasifikasi.

**Jaminan Kualitas**

Kami menerapkan beberapa prosedur untuk mencegah kegagalan model di masa depan:

- **Cek Kebocoran Data:** Kami memastikan kolom target (`emotion` dan `emotion_encoded`) tidak masuk ke dalam rangkaian fitur input. Ini langkah vital supaya model tidak curang saat proses training.

- **Struktur Final:** Dataset kini memiliki format lengkap yang mencakup ID, teks asli, teks bersih, fitur numerik tambahan, serta label target yang sudah siap dipakai.

- **Penyimpanan:** File hasil akhir sudah tersimpan di `curhat_nlp_preprocessed.csv`.

**Ringkasan**

Ringkasnya, kami sudah memiliki total **10.000 baris** data yang bersih dengan kombinasi fitur teks dan numerik. Data ini sudah siap untuk diuji ke berbagai model NLP atau klasifikasi. Langkah selanjutnya adalah melatih model untuk mendeteksi emosi mahasiswa berdasarkan teks yang ada.